# LangChain Memory And Prompt Templates
## RAG



Model used throughout: `gpt-4o-mini`

---
## SECTION 0 - Installation and Setup

In [1]:
import subprocess
import sys

# Minimal dependencies required for this notebook (Memory + Prompt Templates)
packages = [
    "openai",
    "langchain",
    "langchain-openai",
    "langchain-community", # Added for broader compatibility with LangChain components
    "python-dotenv",
]

print("Installing required dependencies...")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages,
    check=True,
)

print("Dependencies installed successfully")

Installing required dependencies...
Dependencies installed successfully


In [2]:
import os

# Set your API keys here directly or via environment
# Replace the placeholder strings with your actual keys

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"          # <-- replace with your OpenAI key
os.environ["LANGCHAIN_API_KEY"] = ""      # <-- replace with your LangSmith key
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "langchain-complete-guide"

# Optional: weather API key for real weather data (we will mock if not set)
os.environ["OPENWEATHER_API_KEY"] = ""             # <-- optional, leave blank to use mock

print("[SETUP] Environment variables configured.")
print(f"[SETUP] OpenAI key set: {'Yes' if os.environ.get('OPENAI_API_KEY', '').startswith('sk-') else 'No - please update'}")
print(f"[SETUP] LangSmith key set: {'Yes' if os.environ.get('LANGCHAIN_API_KEY', '').startswith('lsv2') else 'No - tracing will be disabled'}")

[SETUP] Environment variables configured.
[SETUP] OpenAI key set: Yes
[SETUP] LangSmith key set: No - tracing will be disabled


In [3]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)  # won't overwrite an already-set env var

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f'✅ API key loaded: sk-...{api_key[-4:]}')
else:
    print('⚠️  OPENAI_API_KEY not found.')
    print('   Create a .env file in this folder with: OPENAI_API_KEY=sk-...')

from openai import OpenAI
client = OpenAI()  # automatically reads OPENAI_API_KEY from environment
print('\n✅ OpenAI client initialized')


✅ API key loaded: sk-...VckA

✅ OpenAI client initialized


In [4]:
!pip install langchain-classic langchain-openai


[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: C:\Program Files\Python311\python.exe -m pip install --upgrade pip


---
## SECTION 0.5 - Different Types of Memory in LangChain

LangChain memory helps a chatbot keep context across turns.

Common memory types:
1. ConversationBufferMemory: stores full chat history.
2. ConversationBufferWindowMemory: stores only the last K interactions.
3. ConversationSummaryMemory: keeps a running summary using an LLM.

Below is an example implementation of all three memory types using the same model (`gpt-4o-mini`).

In [ ]:
import os
import langchain_classic
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

print(f"LangChain Classic version: {langchain_classic.__version__}")

def run_demo(chain, name):
    print(f"\n{name}")
    print("-" * len(name))

    prompts = [
        "My name is Sara and I live in Karachi.",
        "I like Python and machine learning.",
        "What is my name and where do I live?",
    ]

    for i, prompt in enumerate(prompts, start=1):
        try:
            response = chain.invoke({"input": prompt})
            print(f"Turn {i} user: {prompt}")
            print(f"Turn {i} bot : {response['response']}\n")
        except Exception as exc:
            print(f"Turn {i} user: {prompt}")
            print(f"Error: {exc}")
            return False
    return True


api_key = os.getenv("OPENAI_API_KEY", "")

if not api_key.startswith("sk-"):
    print("⚠️  OPENAI_API_KEY appears missing or invalid.")
else:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    buffer_chain = ConversationChain(
        llm=llm,
        memory=ConversationBufferMemory(),
        verbose=False,
    )

    run_demo(buffer_chain, "ConversationBufferMemory (full history)")

LangChain Classic version: 1.0.4


C:\Users\Administrator\AppData\Local\Temp\2\ipykernel_4368\2461899331.py:40: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory(),
C:\Users\Administrator\AppData\Local\Temp\2\ipykernel_4368\2461899331.py:38: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  buffer_chain = ConversationChain(
c:\python310\lib\site-packages\langsmith\client.py:618: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(



ConversationBufferMemory (full history)
---------------------------------------


Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019dcdb6-04d4-7560-b0b1-87ad3abc0372,id=019dcdb6-04d4-7560-b0b1-87ad3abc0372; trace=019dcdb6-04d4-7560-b0b1-87ad3abc0372,id=019dcdb6-0552-71e0-aece-eea981070ee2


Turn 1 user: My name is Sara and I live in Karachi.
Turn 1 bot : Hello, Sara! It's great to meet you. Karachi is such a vibrant city with a rich history and diverse culture. Did you know it's the largest city in Pakistan and has a bustling port? What do you enjoy most about living there?



Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019dcdb6-04d4-7560-b0b1-87ad3abc0372,id=019dcdb6-0552-71e0-aece-eea981070ee2; trace=019dcdb6-04d4-7560-b0b1-87ad3abc0372,id=019dcdb6-04d4-7560-b0b1-87ad3abc0372; trace=019dcdb6-0a2b-75b0-9311-f73999103417,id=019dcdb6-0a2b-75b0-9311-f73999103417; trace=019dcdb6-0a2b-75b0-9311-f73999103417,id=019dcdb6-0a2c-7b72-a32e-3a23c4e2353d


Turn 2 user: I like Python and machine learning.
Turn 2 bot : That's awesome, Sara! Python is such a versatile language, and it's a favorite among many in the data science and machine learning communities. With libraries like TensorFlow, PyTorch, and scikit-learn, you can do so much with machine learning. Are you working on any specific projects or areas within machine learning that you're particularly excited about?



Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019dcdb6-0a2b-75b0-9311-f73999103417,id=019dcdb6-0a2c-7b72-a32e-3a23c4e2353d; trace=019dcdb6-0a2b-75b0-9311-f73999103417,id=019dcdb6-0a2b-75b0-9311-f73999103417; trace=019dcdb6-1136-7a02-811a-f31ccc6ce0f3,id=019dcdb6-1136-7a02-811a-f31ccc6ce0f3; trace=019dcdb6-1136-7a02-811a-f31ccc6ce0f3,id=019dcdb6-1138-79f2-b04a-dceb5352029f


Turn 3 user: What is my name and where do I live?
Turn 3 bot : Your name is Sara, and you live in Karachi. It's nice to have this conversation with you! If you have any more questions or topics you'd like to discuss, feel free to share!



Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019dcdb6-1136-7a02-811a-f31ccc6ce0f3,id=019dcdb6-1138-79f2-b04a-dceb5352029f; trace=019dcdb6-1136-7a02-811a-f31ccc6ce0f3,id=019dcdb6-1136-7a02-811a-f31ccc6ce0f3


---
## SECTION 1 - Various Prompt Template Usages in LangChain (E-commerce)

In this section, we use an e-commerce assistant example to demonstrate multiple LangChain prompt template patterns:
1. PromptTemplate (single text prompt)
2. ChatPromptTemplate (system + human structure)
3. FewShotPromptTemplate (example-driven behavior)
4. ChatPromptTemplate with MessagesPlaceholder (conversation history)
5. PromptTemplate with partial variables (reusable constants)

In [6]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotPromptTemplate,
    MessagesPlaceholder,
    PromptTemplate,
)

print("SECTION 1: Prompt template usage for an e-commerce assistant\n")

# ---------------------------------------------------------------
# 1) PromptTemplate: Single string template
# ---------------------------------------------------------------
product_pitch_template = PromptTemplate.from_template(
    """
You are an e-commerce copywriter.
Create a short product pitch for:
- Product: {product_name}
- Category: {category}
- Price: ${price}
- Customer segment: {customer_segment}

Keep it under 50 words and include one CTA.
""".strip()
)

formatted_pitch_prompt = product_pitch_template.format(
    product_name="Noise-Cancelling Headphones",
    category="Electronics",
    price="129",
    customer_segment="College students",
)

print("[1] PromptTemplate output:\n")
print(formatted_pitch_prompt)
print("\n" + "=" * 80 + "\n")

# ---------------------------------------------------------------
# 2) ChatPromptTemplate: Role-based prompt structure
# ---------------------------------------------------------------
order_help_chat_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an e-commerce support agent. Be polite, accurate, and concise.",
        ),
        (
            "human",
            "My order {order_id} has not arrived. Current status is '{status}'. What should I do next?",
        ),
    ]
)

chat_messages = order_help_chat_template.format_messages(
    order_id="EC-92317",
    status="In transit for 8 days",
)

print("[2] ChatPromptTemplate messages:\n")
for msg in chat_messages:
    print(f"{msg.type.upper()}: {msg.content}")
print("\n" + "=" * 80 + "\n")

# ---------------------------------------------------------------
# 3) FewShotPromptTemplate: Learn from examples
# ---------------------------------------------------------------
intent_examples = [
    {"query": "Where is my package?", "intent": "order_tracking"},
    {"query": "I want to return my shoes", "intent": "return_request"},
    {"query": "Do you have this in black?", "intent": "product_availability"},
]

example_template = PromptTemplate.from_template(
    "Customer query: {query}\nIntent: {intent}"
)

intent_classifier_prompt = FewShotPromptTemplate(
    examples=intent_examples,
    example_prompt=example_template,
    prefix="Classify the customer's intent into one short label.",
    suffix="Customer query: {new_query}\nIntent:",
    input_variables=["new_query"],
)

formatted_few_shot_prompt = intent_classifier_prompt.format(
    new_query="Can I cancel order EC-555 right now?"
)

print("[3] FewShotPromptTemplate output:\n")
print(formatted_few_shot_prompt)
print("\n" + "=" * 80 + "\n")

# ---------------------------------------------------------------
# 4) MessagesPlaceholder: Inject prior conversation dynamically
# ---------------------------------------------------------------
history_aware_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a shopping assistant. Use conversation history to avoid repeating questions.",
        ),
        MessagesPlaceholder(variable_name="conversation_history"),
        ("human", "Based on my previous messages, suggest one suitable product."),
    ]
)

history_messages = [
    HumanMessage(content="I need running shoes under $100."),
    AIMessage(content="Do you prefer road running or trail running?"),
    HumanMessage(content="Road running, size 42."),
]

formatted_history_messages = history_aware_template.format_messages(
    conversation_history=history_messages
)

print("[4] ChatPromptTemplate + MessagesPlaceholder messages:\n")
for msg in formatted_history_messages:
    print(f"{msg.type.upper()}: {msg.content}")
print("\n" + "=" * 80 + "\n")

# ---------------------------------------------------------------
# 5) Partial variables: Reuse shared context defaults
# ---------------------------------------------------------------
policy_template = PromptTemplate(
    template=(
        "Store policy for {store_name}:\n"
        "Return window: {return_window_days} days\n"
        "Question: {question}\n"
        "Answer in 2 lines."
    ),
    input_variables=["store_name", "question"],
    partial_variables={"return_window_days": 30},
)

formatted_policy_prompt = policy_template.format(
    store_name="TrendKart",
    question="Can I return a jacket bought 20 days ago?",
)

print("[5] PromptTemplate with partial variables output:\n")
print(formatted_policy_prompt)


SECTION 1: Prompt template usage for an e-commerce assistant

[1] PromptTemplate output:

You are an e-commerce copywriter.
Create a short product pitch for:
- Product: Noise-Cancelling Headphones
- Category: Electronics
- Price: $129
- Customer segment: College students

Keep it under 50 words and include one CTA.


[2] ChatPromptTemplate messages:

SYSTEM: You are an e-commerce support agent. Be polite, accurate, and concise.
HUMAN: My order EC-92317 has not arrived. Current status is 'In transit for 8 days'. What should I do next?


[3] FewShotPromptTemplate output:

Classify the customer's intent into one short label.

Customer query: Where is my package?
Intent: order_tracking

Customer query: I want to return my shoes
Intent: return_request

Customer query: Do you have this in black?
Intent: product_availability

Customer query: Can I cancel order EC-555 right now?
Intent:


[4] ChatPromptTemplate + MessagesPlaceholder messages:

SYSTEM: You are a shopping assistant. Use conversa

In [7]:
# Optional: run one template with an LLM if a valid key is available.
import os
from langchain_openai import ChatOpenAI

api_key = os.getenv("OPENAI_API_KEY", "")
if api_key.startswith("sk-") and "..." not in api_key:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    runnable_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an e-commerce assistant."),
            (
                "human",
                "Create a 3-bullet response for customer issue: {issue}. Include one next action.",
            ),
        ]
    )

    response = (runnable_prompt | llm).invoke(
        {"issue": "My exchange request is pending for 5 days."}
    )
    print("LLM response:\n")
    print(response.content)
else:
    print("Skipped LLM invocation: set a valid OPENAI_API_KEY to run this demo.")


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019dcdb6-9ac6-7b91-b9c5-8b91c92f0b96,id=019dcdb6-9ac6-7b91-b9c5-8b91c92f0b96; trace=019dcdb6-9ac6-7b91-b9c5-8b91c92f0b96,id=019dcdb6-9ad9-7482-8194-d02ff3b1e1b7; trace=019dcdb6-9ac6-7b91-b9c5-8b91c92f0b96,id=019dcdb6-9ad9-7482-8194-d02ff3b1e1b7; trace=019dcdb6-9ac6-7b91-b9c5-8b91c92f0b96,id=019dcdb6-9adf-7590-bf99-2e0b14a408d4


LLM response:

- We apologize for the delay in processing your exchange request and understand how important it is to resolve this quickly.
- Our team is currently reviewing your request, and we appreciate your patience during this time.
- As a next step, we will escalate your request to our exchange department for immediate attention and will provide you with an update within the next 24 hours. Thank you for your understanding!


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019dcdb6-9ac6-7b91-b9c5-8b91c92f0b96,id=019dcdb6-9adf-7590-bf99-2e0b14a408d4; trace=019dcdb6-9ac6-7b91-b9c5-8b91c92f0b96,id=019dcdb6-9ac6-7b91-b9c5-8b91c92f0b96
